<a href="https://colab.research.google.com/github/lbush5355/PoseAI/blob/main/PoseAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Step 1: Environment Setup & Module Synchronization

from google.colab import drive
import os
import sys
import shutil
import logging
import glob
import collections
from datetime import datetime
import requests

# Mount Google Drive (handle if already mounted)
try:
    drive.mount('/content/drive', force_remount=False)
except RuntimeError as e:
    if "already mounted" in str(e):
        print("Drive already mounted")
    else:
        raise

# Set the root project directory in Google Drive (used for dataset/, results/, batch_results/)
PROJECT_PATH = "/content/drive/MyDrive/PoseAI" # @param {type:"string"}

# Setup environment paths
FAST_LANE = "/content/fast_lane"
SRC_DIR = f"{FAST_LANE}/src"
BIN_DIR = f"{FAST_LANE}/bin"

# Create required directories
os.makedirs(SRC_DIR, exist_ok=True)
os.makedirs(f"{FAST_LANE}/logs", exist_ok=True)

print("Setting up PoseAI environment...")

# Sync src/ modules from GitHub — single source of truth.
# Local edits → git push → notebook clones fresh on every session.
# Drive's src/ is no longer authoritative.
GIT_REPO = "https://github.com/lbush5355/PoseAI.git" # @param {type:"string"}
GIT_REF = "main" # @param {type:"string"}
REPO_CLONE_DIR = "/tmp/poseai_repo"

print(f"Cloning {GIT_REPO} @ {GIT_REF}...")
if os.path.exists(REPO_CLONE_DIR):
    shutil.rmtree(REPO_CLONE_DIR)
clone_rc = os.system(
    f"git clone -q --depth 1 --branch {GIT_REF} {GIT_REPO} {REPO_CLONE_DIR}"
)
if clone_rc != 0:
    raise RuntimeError(f"git clone failed (rc={clone_rc}): {GIT_REPO} ref={GIT_REF}")

repo_src = f"{REPO_CLONE_DIR}/src"
for fname in sorted(os.listdir(repo_src)):
    if fname.endswith(".py"):
        shutil.copy2(f"{repo_src}/{fname}", f"{SRC_DIR}/{fname}")
        print(f"  Copied {fname}")

head_sha = os.popen(f"git -C {REPO_CLONE_DIR} rev-parse --short HEAD").read().strip()
print(f"Modules synced @ {GIT_REF} ({head_sha})")

# Configure Python path and environment
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
os.environ["PATH"] += f":{BIN_DIR}:/usr/local/bin"

# Install required dependencies
print("\nInstalling Python dependencies...")
os.system('pip install -q rdkit openbabel-wheel py3Dmol hdbscan -q')
print("Dependencies installed")

# Import 3rd party libraries after installation
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign, Draw
import py3Dmol

# Install 32-bit architecture and libraries for LeDock
print("\nInstalling system dependencies (32-bit libraries for LeDock)...")
os.system('sudo dpkg --add-architecture i386')
os.system('sudo apt-get update -qq')
os.system('sudo apt-get install -y -qq libc6:i386 libncurses5:i386 libstdc++6:i386 zlib1g:i386')
print("System dependencies installed")

# Build fpocket from source
print("\nVerifying fpocket installation...")
if not os.path.exists('/usr/local/bin/fpocket'):
    print("Building fpocket from source...")
    os.system('git clone -q https://github.com/Discngine/fpocket.git /tmp/fpocket')
    os.system('cd /tmp/fpocket && make -s && sudo make install -s')
    print("fpocket compiled and installed")
else:
    print("fpocket already installed")

# Configure logging early so download_and_verify_binary's poseai.utils
# logger emits to the console during the binary verification step.
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(name)-24s | %(levelname)-8s | %(message)s'
)
logger = logging.getLogger("poseai")

# Download and verify docking engine binaries.
# download_and_verify_binary fails loudly on HTML error pages, partial
# downloads, or wrong-architecture binaries — instead of silently writing
# garbage to disk and breaking later in subprocess.Popen.
from utils import download_and_verify_binary

print("\nDownloading and verifying docking engine binaries...")
os.makedirs(BIN_DIR, exist_ok=True)

download_and_verify_binary(
    "https://sourceforge.net/projects/smina/files/smina.static/download",
    f"{BIN_DIR}/smina", "smina"
)
download_and_verify_binary(
    "https://github.com/gnina/gnina/releases/download/v1.1/gnina",
    f"{BIN_DIR}/gnina", "gnina"
)
download_and_verify_binary(
    "https://www.lephar.com/download/ledock_linux_x86",
    f"{BIN_DIR}/ledock", "ledock"
)
download_and_verify_binary(
    "https://www.lephar.com/download/lepro_linux_x86",
    f"{BIN_DIR}/lepro", "lepro"
)

print("All binaries verified.")

# Import and validate configuration & custom modules
from config import get_config
config = get_config()
config.validate_all()

from preprocessor import (
    ProteinLigandPrep, fetch_rcsb_smiles, fetch_rcsb_ideal_sdf,
    fetch_rcsb_entry_ligand_codes,
    extract_ligand_code_candidates_from_mol2, get_ligand_centroid,
)
from docking import EnsembleManager, EngineType
from consensus import ConsensusAnalyzer
from visualizer import DockingVisualizer

logger.info("Environment initialization complete")
logger.info(f"Modules directory: {SRC_DIR}")
logger.info(f"Binaries directory: {BIN_DIR}")
logger.info(f"Source commit: {head_sha} ({GIT_REF})")

# Auto-generate environment.yml for local repository completeness
env_yml = """name: poseai
channels:
  - conda-forge
  - defaults
dependencies:
  - python>=3.8
  - rdkit
  - openbabel
  - numpy
  - pandas
  - hdbscan
  - pip
  - pip:
    - py3Dmol
"""
with open(f"{PROJECT_PATH}/environment.yml", "w") as f:
    f.write(env_yml)
logger.info(f"Generated environment.yml at {PROJECT_PATH}")


In [ ]:
# @title Step 2: PoseAI Pipeline Configuration

TARGET_PDB = "1hsg" # @param {type:"string"}
LIGAND_CODE = "MK1" # @param {type:"string"}
LIGAND_SMILES = "CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCC[C@@H]2CN1C[C@@H](O)[C@H](Cc3ccccc3)NC(=O)c4cnccn4" # @param {type:"string"}
# (Leave LIGAND_SMILES empty to dynamically fetch reference from RCSB PDB)

EXHAUSTIVENESS = 32 # @param {type:"slider", min:1, max:64}
POSES_PER_ENGINE = 20 # @param {type:"number"}
BOX_PADDING = 5 # @param {type:"slider", min:5, max:20}
RMSD_THRESHOLD = 2.5 # @param {type:"number"}
TIMEOUT_SECONDS = 3600 # @param {type:"slider", min:600, max:7200}

USE_GNINA = True # @param {type:"boolean"}
USE_SMINA = True # @param {type:"boolean"}
USE_LEDOCK = True # @param {type:"boolean"}

print("✓ Parameters loaded from Step 2")


In [ ]:
# =============================================================================
# @title Step 3: PoseAI Unified Execution Pipeline (v1.0)
# =============================================================================

# Read live parameters from Cell 2 (globals)
TARGET_PDB = globals().get('TARGET_PDB', '1iep').lower()
LIGAND_CODE = globals().get('LIGAND_CODE', 'STI')
LIGAND_SMILES = globals().get('LIGAND_SMILES', None)

# Set correct result directories for the current target
RESULTS_DIR = f"/content/fast_lane/results/{TARGET_PDB.upper()}"
DRIVE_RESULTS = f"{PROJECT_PATH}/results/{TARGET_PDB.upper()}"

print("\n" + "="*70)
print("POSEAI PIPELINE v1.0 — INITIAL RELEASE")
print("="*70)

# Master Topology / Reference Fetching (Using newly migrated function)
if not LIGAND_SMILES or LIGAND_SMILES == "":
    print(f"\nFetching reference SMILES for {LIGAND_CODE}...")
    LIGAND_SMILES = fetch_rcsb_smiles(LIGAND_CODE)
    if LIGAND_SMILES:
        print(f"✓ Dynamically loaded SMILES: {LIGAND_SMILES}")
    else:
        print(f"⚠ Could not find SMILES for {LIGAND_CODE}. Consensus auto-detection may fail.")

print("\nPipeline Parameters")
print(f"  PDB ID:              {TARGET_PDB.upper()}")
print(f"  Ligand Code:         {LIGAND_CODE}")
print(f"  Engines:             {[e for e in ['GNINA', 'SMINA', 'LEDOCK'] if globals().get(f'USE_{e}', True)]}")

# STAGE 1: Preprocessing
print("\n" + "="*70)
print("STAGE 1: Structure Preparation")
print("="*70)
prep = ProteinLigandPrep(TARGET_PDB)
if not prep.fetch_structure() or not prep.prepare_receptor() or not prep.isolate_ligand(LIGAND_CODE):
    raise RuntimeError("Failed in preprocessing stage")
print(f"✓ Receptor: {prep.receptor_pdbqt}\n✓ Ligand:   {prep.ligand_mol2}")

# STAGE 2: Pocket Detection (Using newly migrated function)
print("\n" + "="*70)
print("STAGE 2: Binding Pocket Detection")
print("="*70)
center, size = get_ligand_centroid(prep.ligand_mol2, padding=BOX_PADDING)
print(f"✓ Search center: ({center[0]:.1f}, {center[1]:.1f}, {center[2]:.1f})")
print(f"✓ Search size:   ({size[0]:.1f}, {size[1]:.1f}, {size[2]:.1f})")

# STAGE 3: Ensemble Docking
print("\n" + "="*70)
print("STAGE 3: Ensemble Docking")
print("="*70)
engines_to_run = []
if USE_GNINA: engines_to_run.append(EngineType.GNINA)
if USE_SMINA: engines_to_run.append(EngineType.SMINA)
if USE_LEDOCK: engines_to_run.append(EngineType.LEDOCK)

mgr = EnsembleManager()
results = mgr.run_ensemble(
    prep.receptor_pdbqt, prep.ligand_pdbqt, center, size, RESULTS_DIR,
    exhaustiveness=EXHAUSTIVENESS, num_modes=POSES_PER_ENGINE, receptor_pdb=prep.receptor_pdb,
    ligand_mol2=prep.ligand_mol2, n_ledock_poses=POSES_PER_ENGINE, timeout=TIMEOUT_SECONDS, engines=engines_to_run
)
print(f"✓ Ensemble complete: {sum(1 for r in results if r.success)}/{len(results)} successful")

# STAGE 4: Consensus Clustering
print("\n" + "="*70)
print("STAGE 4: Consensus Clustering")
print("="*70)
# Fetch the canonical ideal-SDF topology template (preferred over
# obabel-perceived crystal mol2 for AssignBondOrdersFromTemplate).
ideal_sdf_path = fetch_rcsb_ideal_sdf(LIGAND_CODE, prep.work_dir)
analyzer = ConsensusAnalyzer(
    work_dir="/content/fast_lane",
    rmsd_threshold=RMSD_THRESHOLD,
    ligand_smiles=LIGAND_SMILES,
    reference_ligand_path=prep.ligand_mol2,
    topology_template_path=ideal_sdf_path,
)
try:
    df = analyzer.analyze_ensemble(results)

    # Pose breakdown log
    if hasattr(analyzer, '_metadata'):
        engines_loaded = [meta.get('engine', 'UNKNOWN') for meta in analyzer._metadata]
        counts = collections.Counter(engines_loaded)
        print(f"\n✓ Poses loaded into Consensus: {len(analyzer.all_poses)} total")
        for eng, c in counts.items():
            print(f"    - {eng}: {c} poses")
        print()

    if df is not None and not df.empty:
        print(f"✓ Clustering complete: {len(df)} consensus cluster(s)\n")
        print(df.to_string(index=False))

        # STAGE 5: Scoring & Visualization
        print("\nSTAGE 5: Visualization")
        best_cluster = df.iloc[0]
        print(f"✓ Best cluster: {int(best_cluster['Cluster'])} | Confidence: {analyzer.get_confidence_score(df)}")

        viz = DockingVisualizer(prep.receptor_pdb)
        viz.create_interactive_view()
        indices = np.where(analyzer.cluster_labels == best_cluster['Cluster'])[0].tolist()
        viz.add_consensus_cluster(analyzer.all_poses, indices, label="Consensus")
        viz.show()

        # Export
        os.makedirs(RESULTS_DIR, exist_ok=True)
        df.to_csv(os.path.join(RESULTS_DIR, "cluster_summary.csv"), index=False)
        shutil.copytree(RESULTS_DIR, DRIVE_RESULTS, dirs_exist_ok=True)
        print("✓ Results archived.")
    else:
        print("✗ No consensus clusters found.")
except Exception as e:
    print(f"✗ Consensus analysis failed: {e}")

print("\n" + "="*70 + "\nPIPELINE COMPLETE\n" + "="*70)

In [ ]:
# =============================================================================
# @title Step 4: Validation (Native RMSD & Overlay)
# =============================================================================

print("\n" + "="*70)
print("STAGE 4: VALIDATION (Native RMSD Calculation)")
print("="*70)

from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign
import numpy as np
from visualizer import DockingVisualizer
from consensus import calculate_native_rmsd

try:
    # Get the best pose from the top consensus cluster
    best_cluster_id = int(df.iloc[0]['Cluster'])
    best_indices = np.where(analyzer.cluster_labels == best_cluster_id)[0]

    # Take the representative (first) pose of the consensus cluster
    top_pose_mol = analyzer.all_poses[best_indices[0]]

    # Calculate In-Place RMSD using our robust function
    rmsd = calculate_native_rmsd(top_pose_mol, prep.ligand_mol2, reference_smiles=LIGAND_SMILES)

    print(f"\u2713 Reference Ligand loaded: {prep.ligand_mol2}")
    print(f"\u2713 Top Consensus Pose extracted from Cluster {best_cluster_id}")
    print(f"\nStrict In-Place Heavy-Atom RMSD: {rmsd:.3f} \u00C5")

    if rmsd <= 2.0:
        print("  -> EXCELLENT: Pose is within the universally accepted 2.0 \u00C5 threshold for a 'success'.")
    elif rmsd <= 3.0:
        print("  -> ACCEPTABLE: Pose represents the correct general binding mode.")
    else:
        print("  -> POOR: Pose deviates significantly from the native crystal structure.")

    # Visualize the overlay
    print("\nRendering Native Overlay...")
    viz_val = DockingVisualizer(prep.receptor_pdb)
    viz_val.create_interactive_view()

    # Add Reference in Green
    with open(prep.ligand_mol2, 'r') as f:
        viz_val.view.addModel(f.read(), 'mol2')
        viz_val.view.setStyle({'model': -1}, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.15}})

    # Add Predicted Pose in Cyan
    pose_block = Chem.MolToMolBlock(top_pose_mol)
    viz_val.view.addModel(pose_block, 'mol')
    viz_val.view.setStyle({'model': -1}, {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.15}})

    viz_val.view.zoomTo()
    viz_val.show()
    print("\u25a0 Green: Native Crystal Ligand  |  \u25a0 Cyan: Top Predicted Consensus Pose")

except Exception as e:
    print(f"\n\u2717 Validation failed: {e}")
    print("Make sure Step 3 completed successfully and variables 'prep', 'df', and 'analyzer' are in memory.")

In [ ]:
# =============================================================================
# @title Step 5: Local Dataset Batch Validation
# =============================================================================

DATASET_DIR = f"{PROJECT_PATH}/dataset"
BATCH_RESULTS_DIR = f"{PROJECT_PATH}/batch_results"
os.makedirs(BATCH_RESULTS_DIR, exist_ok=True)

print("="*70)
print("STAGE 5: LOCAL DATASET BATCH VALIDATION")
print("="*70)

results_list = []

if not os.path.exists(DATASET_DIR) or not os.listdir(DATASET_DIR):
    print(f"Dataset directory not found or empty: {DATASET_DIR}")
    print("Please ensure your target folders (e.g., '1hsg') are uploaded.")
else:
    target_folders = [f for f in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, f))]
    print(f"Found {len(target_folders)} targets in {DATASET_DIR}\n")

    for target_id in target_folders:
        target_id = target_id.lower()
        target_path = os.path.join(DATASET_DIR, target_id)
        print(f"Processing Target: {target_id.upper()}")
        print("-"*30)

        # Standard PDBbind/CASF naming convention
        receptor_file = os.path.join(target_path, f"{target_id}_protein.pdb")
        ligand_file = os.path.join(target_path, f"{target_id}_ligand.mol2")

        if not os.path.exists(receptor_file) or not os.path.exists(ligand_file):
            print(f"  -> Skipping: Missing receptor_protein.pdb or ligand.mol2 in {target_path}\n")
            results_list.append({"Target": target_id.upper(), "Status": "Missing Files", "RMSD": None, "Confidence": None, "Composition": None})
            continue

        try:
            # 1. Structure Preparation (using local files)
            # We use obabel to generate the required PDBQT files locally
            receptor_pdbqt = os.path.join(target_path, f"{target_id}_protein.pdbqt")
            ligand_pdbqt = os.path.join(target_path, f"{target_id}_ligand.pdbqt")

            # Skip regeneration if PDBQT files already exist (idempotent re-runs)
            if not os.path.exists(receptor_pdbqt):
                print("  -> Generating receptor PDBQT...")
                os.system(f"obabel {receptor_file} -O {receptor_pdbqt} -xr > /dev/null 2>&1")
            if not os.path.exists(ligand_pdbqt):
                print("  -> Generating ligand PDBQT...")
                os.system(f"obabel {ligand_file} -O {ligand_pdbqt} -p 7.4 > /dev/null 2>&1")

            # 2. Pocket Detection
            center, size = get_ligand_centroid(ligand_file, padding=BOX_PADDING)

            # 3. Ensemble Docking
            engines_to_run = []
            if USE_GNINA: engines_to_run.append(EngineType.GNINA)
            if USE_SMINA: engines_to_run.append(EngineType.SMINA)
            if USE_LEDOCK: engines_to_run.append(EngineType.LEDOCK)

            target_results_dir = os.path.join(BATCH_RESULTS_DIR, target_id.upper())
            os.makedirs(target_results_dir, exist_ok=True)

            print("  -> Running Docking Ensemble...")
            mgr = EnsembleManager()
            docking_results = mgr.run_ensemble(
                receptor_pdbqt, ligand_pdbqt, center, size, target_results_dir,
                exhaustiveness=EXHAUSTIVENESS, num_modes=POSES_PER_ENGINE, receptor_pdb=receptor_file,
                ligand_mol2=ligand_file, n_ledock_poses=POSES_PER_ENGINE, timeout=TIMEOUT_SECONDS, engines=engines_to_run
            )

            # 4. Consensus Clustering
            print("  -> Analyzing Consensus...")
            native_mol = Chem.MolFromMol2File(ligand_file, sanitize=False)
            native_smiles = Chem.MolToSmiles(native_mol) if native_mol else None

            # Topology source for AssignBondOrdersFromTemplate:
            # primary = RCSB GraphQL lookup (authoritative);
            # fallback = mol2 substructure-name candidates;
            # iterate candidates and use the first ideal SDF that downloads.
            candidates = fetch_rcsb_entry_ligand_codes(target_id)
            if not candidates:
                candidates = extract_ligand_code_candidates_from_mol2(ligand_file)
            ideal_sdf_path = None
            for code in candidates:
                path = fetch_rcsb_ideal_sdf(code, target_path)
                if path:
                    ideal_sdf_path = path
                    break
            analyzer = ConsensusAnalyzer(
                work_dir="/content/fast_lane",
                rmsd_threshold=RMSD_THRESHOLD,
                ligand_smiles=native_smiles,
                reference_ligand_path=ligand_file,
                topology_template_path=ideal_sdf_path,
            )
            df = analyzer.analyze_ensemble(docking_results)

            if df is not None and not df.empty:
                best_cluster_id = int(df.iloc[0]['Cluster'])
                best_indices = np.where(analyzer.cluster_labels == best_cluster_id)[0]
                top_pose_mol = Chem.RemoveHs(analyzer.all_poses[best_indices[0]])

                # Calculate RMSD
                # Reference: crystal positions from native_mol, bond orders
                # from analyzer.master_ref (ideal SDF when available, else crystal).
                # Both top_pose_mol and ref_mol end up with isomorphic graphs.
                try:
                    ref_mol = Chem.RemoveHs(
                        AllChem.AssignBondOrdersFromTemplate(analyzer.master_ref, native_mol)
                    )
                except (ValueError, RuntimeError):
                    ref_mol = analyzer.master_ref  # last-resort fallback
                rmsd = rdMolAlign.CalcRMS(top_pose_mol, ref_mol)
                print(f"  -> Native RMSD: {rmsd:.3f} Å")

                # Determine Validation Status
                if rmsd <= 2.0:
                    status = "Success"
                    print("  -> Validation: SUCCESS (RMSD <= 2.0 Å)")
                elif rmsd <= 3.0:
                    status = "Acceptable"
                    print("  -> Validation: ACCEPTABLE (RMSD <= 3.0 Å)")
                else:
                    status = "Poor"
                    print("  -> Validation: FAILED (RMSD > 3.0 Å)")

                # Get Confidence Score
                conf_score = analyzer.get_confidence_score(df)
                print(f"  -> Predicted Confidence: {conf_score}")

                # Extract Engine Composition for Top Cluster
                composition_str = "Unknown"
                if hasattr(analyzer, '_metadata'):
                    engine_counts = collections.Counter()
                    for idx in best_indices:
                        eng = analyzer._metadata[idx].get('engine', 'UNKNOWN')
                        engine_counts[eng] += 1

                    print(f"  -> Top Cluster Engine Breakdown:")
                    breakdown_list = []
                    for eng, count in engine_counts.items():
                        print(f"       - {eng}: {count} poses")
                        breakdown_list.append(f"{eng}:{count}")
                    composition_str = ", ".join(breakdown_list)

                results_list.append({"Target": target_id.upper(), "Status": status, "RMSD": rmsd, "Confidence": conf_score, "Composition": composition_str})

                # Generate 3D HTML View
                html_path = os.path.join(target_results_dir, f"{target_id.upper()}_view.html")
                view = py3Dmol.view(width=800, height=600)
                view.setBackgroundColor('white')

                # Protein
                with open(receptor_file, 'r') as f: view.addModel(f.read(), 'pdb')
                view.setStyle({'model': 0}, {'cartoon': {'color': 'lightgray'}})

                # Native Ligand
                with open(ligand_file, 'r') as f: view.addModel(f.read(), 'mol2')
                view.setStyle({'model': 1}, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.15}})

                # Predicted Pose
                view.addModel(Chem.MolToMolBlock(top_pose_mol), 'mol')
                view.setStyle({'model': 2}, {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.15}})

                view.zoomTo({'model': 1})
                with open(html_path, 'w') as f: f.write(view._make_html())

            else:
                print("  -> No consensus clusters found.")
                results_list.append({"Target": target_id.upper(), "Status": "No Clusters", "RMSD": None, "Confidence": None, "Composition": None})

        except Exception as e:
            print(f"  -> Failed: {e}")
            results_list.append({"Target": target_id.upper(), "Status": "Error", "RMSD": None, "Confidence": None, "Composition": None})

        print("\n")

    # Display and Save summary
    if results_list:
        summary_df = pd.DataFrame(results_list)
        summary_csv = os.path.join(BATCH_RESULTS_DIR, "batch_summary.csv")
        summary_df.to_csv(summary_csv, index=False)
        print(f"Batch validation complete! Summary saved to: {summary_csv}\n")
        display(summary_df)
